# 03 — Module C: Customer Segmentation (Clustering)
Discover borrower segments using unsupervised learning.

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = "/Users/nando/PycharmProjects/PythonProject/SmartLender"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import plotly.express as px
import joblib

from src.data.loader import load_raw_data
from src.config import *

%matplotlib inline

## Load and Prepare Data

In [ ]:
df = load_raw_data(sample_size=200000)

# Use only numerical features for clustering
num_cols = [c for c in NUMERICAL_FEATURES if c in df.columns]
X = df[num_cols].dropna()
print(f"Clustering data shape: {X.shape}")

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Elbow Curve and Silhouette Scores

In [ ]:
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=10000)
    silhouettes.append(sil)
    print(f"k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(k_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Curve')

ax2.plot(k_range, silhouettes, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Scores')

plt.tight_layout()
plt.savefig('results/figures/module_c_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

## Fit Final K-Means

In [ ]:
best_k = 5  # Adjust based on elbow/silhouette analysis above
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df_clustered = X.copy()
df_clustered['cluster'] = cluster_labels
print(f"Cluster distribution:\n{pd.Series(cluster_labels).value_counts().sort_index()}")

## DBSCAN Comparison

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Find optimal eps using k-distance plot
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled[:10000])  # Sample for speed
distances, _ = nn.kneighbors(X_scaled[:10000])
distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.xlabel('Points')
plt.ylabel('5th Nearest Neighbor Distance')
plt.title('K-Distance Plot for DBSCAN eps Selection')
plt.tight_layout()
plt.show()

# Try DBSCAN with a few eps values
for eps in [1.0, 1.5, 2.0, 3.0]:
    db = DBSCAN(eps=eps, min_samples=10)
    db_labels = db.fit_predict(X_scaled[:50000])
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    noise_pct = (db_labels == -1).mean() * 100
    print(f"eps={eps}: {n_clusters} clusters, {noise_pct:.1f}% noise")

## PCA Projection

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig = px.scatter(
    x=X_pca[:, 0], y=X_pca[:, 1],
    color=cluster_labels.astype(str),
    title=f'K-Means Clusters (k={best_k}) — PCA Projection',
    labels={'x': f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)',
            'y': f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)'},
    opacity=0.3,
)
fig.update_layout(height=600)
fig.show()

## t-SNE Projection

In [ ]:
# Sample for t-SNE (slow on full data)
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_scaled), size=min(10000, len(X_scaled)), replace=False
)
X_tsne_input = X_scaled[sample_idx]
labels_sample = cluster_labels[sample_idx]

tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
X_tsne = tsne.fit_transform(X_tsne_input)

fig = px.scatter(
    x=X_tsne[:, 0], y=X_tsne[:, 1],
    color=labels_sample.astype(str),
    title=f'K-Means Clusters (k={best_k}) — t-SNE Projection',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2'},
    opacity=0.3,
)
fig.update_layout(height=600)
fig.show()

## Cluster Profiles

In [ ]:
profile = df_clustered.groupby('cluster').mean().round(2)
print("\nCluster Profiles (Mean Feature Values):")
print(profile.to_string())

# Save for Streamlit
os.makedirs('results/comparison_tables', exist_ok=True)
profile.to_csv('results/comparison_tables/module_c_profiles.csv')

# Save cluster data for Streamlit
cluster_data = pd.DataFrame({
    'pca_1': X_pca[:, 0], 'pca_2': X_pca[:, 1], 'cluster': cluster_labels
})
cluster_data.to_parquet('data/processed/cluster_data.parquet', index=False)

# Save KMeans model and scaler
os.makedirs('models/module_c', exist_ok=True)
joblib.dump(kmeans, 'models/module_c/kmeans.joblib')
joblib.dump(scaler, 'models/module_c/scaler.joblib')
joblib.dump(pca, 'models/module_c/pca.joblib')
print('Saved cluster data and models.')